In [1]:
import random, os
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizerFast, BertModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path

D:\projects\Supervised-Learning-Experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 2026
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
BASE = Path.cwd().parent / "data"

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')

print(f"train {train.shape}, test {test.shape}, label counts {train['label'].value_counts().to_dict()}")
train.head()

train (50, 3), test (686, 3), label counts {0: 25, 1: 25}


,w1,w2,label
0,cash,money,0
1,grab,take,0
2,irrational,rational,1
3,defend,guard,0
4,instructor,teacher,0


In [5]:
tokenizer = BertTokenizerFast.from_pretrained('bert-large-uncased')
bert = BertModel.from_pretrained('bert-large-uncased').eval().to(device)
    
embeddings = bert.get_input_embeddings().weight  

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6869.23it/s]
[transformers] BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def word_vector(word):
    return embeddings[tokenizer.convert_tokens_to_ids(word)]

def pair_features(a, b):
    va, vb = word_vector(a), word_vector(b)
    cosine = torch.cosine_similarity(va, vb, dim=0).item()
    distance = (va - vb).norm().item()
    return [cosine, distance]

X_train = np.array([pair_features(a, b) for a, b in zip(train['w1'], train['w2'])])
X_test = np.array([pair_features(a, b) for a, b in zip(test['w1'], test['w2'])])
y_train = train['label'].values

In [11]:
mean, std = X_train.mean(0), X_train.std(0) 
X_train, X_test = (X_train - mean) / std, (X_test - mean) / std
print(X_train)
# clf = LogisticRegression().fit(X_train, y_train)
clf = RandomForestClassifier().fit(X_train, y_train)


print(f"train accuracy {(clf.predict(X_train) == y_train).mean():.3f}")

[[ 0.26383905 -0.17918635]
 [-1.08286856  0.32712758]
 [ 1.21397103 -0.7502808 ]
 [-0.92426351  0.6916982 ]
 [ 0.59190298 -0.83984271]
 [-1.10417908  0.69536344]
 [ 0.35622043 -0.64034456]
 [-0.4363029  -0.14013955]
 [-0.24300348  0.56090435]
 [ 0.54813019  0.06524634]
 [ 2.15062501 -1.88747618]
 [ 0.72660881 -0.13909293]
 [-0.73244312  0.91061954]
 [-1.62637076  1.6072546 ]
 [-1.11872604  1.00347548]
 [ 1.18838376 -0.42796739]
 [-0.91283518  0.58705078]
 [-0.6838798   1.4800062 ]
 [ 1.12839598 -0.34113973]
 [-1.26832738  1.70729513]
 [ 0.0277951  -0.38788405]
 [-1.31097355  1.92420566]
 [-0.51261416  0.50731462]
 [-0.52564228  0.11738445]
 [ 1.42211444 -1.70755242]
 [-1.20865995  1.29543167]
 [-0.84316626  0.23956818]
 [ 0.36562273  0.21491843]
 [-0.06939273 -1.19551561]
 [-0.21307737  0.2581511 ]
 [ 0.25814445 -0.88109956]
 [ 1.49079036 -1.06558514]
 [ 1.48351461 -0.63965551]
 [ 1.53861019 -1.74234397]
 [ 0.90046179 -1.09561088]
 [-1.01351555  0.73782879]
 [ 1.4297318  -1.05357544]
 

In [8]:
pred = clf.predict(X_test)

submission = pd.DataFrame({'row_id': test['row_id'], 'label': pred})
submission.to_csv(Path.cwd().parent / 'submission.csv', index=False)

In [9]:
print(f"{int(pred.sum())} antonyms predicted of {len(pred)}")
submission.head()

388 antonyms predicted of 686


,row_id,label
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1
